# Local Nonlinear Support Assembly Demo

This notebook demonstrates nonlinear local support assembly with the same Python-side partition and support-patch workflow as the linear demo. Python handles METIS partitioning, support-patch construction, DoF/index bookkeeping, visualization, and verification. C++ receives the patch metadata and assembles core-sized nonlinear residuals and Jacobians over the supplied support elements.


In [ ]:
import math
import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.geom2d import unit_square

from partition.metis import metis_partition_from_fes
from utils.patches import build_support_patches, draw_patch, print_patch_summary

import myassembling
print(myassembling)
print(myassembling.__file__)
print(dir(myassembling))

SetNumThreads(1)

## Mesh, Space, And Nonlinear Form

The nonlinear form is a generic NGSolve `BilinearForm`. The C++ nonlinear assembler does not own the PDE model; it uses the supplied form and current global vector to compute element nonlinear residual and Jacobian contributions.

Here we use the same problem as `demo_nonlinear_local_assembly.py`:

`F(u; v) = int grad(u) . grad(v) dx + int u^3 v dx`.


In [ ]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.15))
fes = H1(mesh, order=2)
u, v = fes.TnT()

a = BilinearForm(fes)
a += (grad(u) * grad(v) + u * u * u * v) * dx

gfu = GridFunction(fes, name="u_current")
gfu.Set((x * (1 - x)) ** 2 + 0.25 * (y * (1 - y)) ** 2)

print("ne =", mesh.ne, "ndof =", fes.ndof)
Draw(gfu, mesh, "u_current")

## METIS Element Partition

As in the linear demo, `partition/metis.py` builds a shared-DoF element graph and calls METIS. `core_partition` is non-overlapping. `overlapping_partition` is available for experiments, but the exact support closure is built separately in `utils/patches.py`.


In [ ]:
core_partition, overlapping_partition, cutcount = metis_partition_from_fes(
    fes,
    nparts=3,
    overlap_width=1,
    free_dofs_only=False,
)

partition = core_partition

print("METIS cutcount =", cutcount)
print("core element counts =", [len(p) for p in core_partition])
print("overlapping element counts =", [len(p) for p in overlapping_partition])

print(f"{core_partition = }")
print(f"{overlapping_partition = }")

## Visualize The Partition

Each plot shows one METIS subdomain. Value `1.0` marks elements in `core_partition[i]`.


In [ ]:
l2 = L2(mesh, order=0)

for i, elements in enumerate(partition):
    omega_i = GridFunction(l2, name=f"Omega_{i}")
    omega_i.vec[:] = 0
    for elnr in elements:
        omega_i.vec[l2.GetDofNrs(ElementId(VOL, elnr))[0]] = 1.0
    Draw(omega_i, mesh, f"subdomain {i}: core=1")


## Build Support Patches In Python

For each selected element set, Python constructs the exact support closure:

- `core_dofs`: all global DoFs appearing on `core_elements`
- `support_elements`: all elements whose DoF list intersects `core_dofs`
- `support_dofs`: all global DoFs appearing on `support_elements`
- `core_in_support`: local indices of `core_dofs` inside `support_dofs`

The nonlinear C++ functions use the same metadata as the linear local matrix/vector functions.


In [ ]:
support_patches = build_support_patches(fes, partition)

for i, patch in enumerate(support_patches):
    print(f"\nOmega_{i}:")
    print_patch_summary(patch)
    for k, g in enumerate(patch.core_dofs):
        assert patch.support_dofs[patch.core_in_support[k]] == g


## Visualize Support Patches

Each plot shows one support patch. Value `1` marks input core elements. Value `2` marks support-only elements that are outside the core but contribute to entries involving the core DoFs.


In [ ]:
for i, patch in enumerate(support_patches):
    draw_patch(mesh, patch, name=f"Omega_{i}: core=1, support-only=2")

## Global Nonlinear Residual And Jacobian

For verification, assemble the global nonlinear residual and Jacobian using NGSolve's nonlinear form machinery:

- `a.AssembleLinearization(gfu.vec)` assembles the Jacobian at the same state, stored as `a.mat`.
- `a.Apply(gfu.vec, res_global)` evaluates the nonlinear residual at the current state.


In [ ]:
# global jacobian matrix
a.AssembleLinearization(gfu.vec)
jac_global = a.mat
# global residual
res_global = gfu.vec.CreateVector()
a.Apply(gfu.vec, res_global)

## Assemble Local Nonlinear Residuals And Jacobians In C++

This follows `demo_nonlinear_local_assembly.py`. For each patch, C++ receives the same five patch fields used by the linear demo. It loops over `support_elements`, computes nonlinear element contributions from the global form and current vector, and returns core-sized local objects:

- `jac_local.mat`: shape `len(core_dofs) x len(core_dofs)`
- `res_local.vec`: length `len(core_dofs)`


In [ ]:
local_jacobians = []
local_residuals = []

for patch in support_patches:
    local_jacobians.append(
        myassembling.MyAssembleLocalNonlinearJacobian(
            fes,
            a,
            gfu.vec,
            patch.core_elements,
            patch.support_elements,
            patch.core_dofs,
            patch.support_dofs,
            patch.core_in_support,
        )
    )
    local_residuals.append(
        myassembling.MyAssembleLocalNonlinearResidual(
            fes,
            a,
            gfu.vec,
            patch.core_elements,
            patch.support_elements,
            patch.core_dofs,
            patch.support_dofs,
            patch.core_in_support,
        )
    )

for i, (patch, jac_local, res_local) in enumerate(zip(support_patches, local_jacobians, local_residuals)):
    print(f"Omega_{i}: residual length =", len(patch.core_dofs),
          "Jacobian shape =", (jac_local.mat.height, jac_local.mat.width),
          "Residual shape =", (res_local.vec.size))

## Verify Against Global Nonlinear Assembly

The local nonlinear residual returned by C++ is already the core vector:

`F_local == F_global[core_dofs]`

The local nonlinear Jacobian returned by C++ is already the core-core matrix:

`J_local == J_global[core_dofs, core_dofs]`

We do not compare full support objects with global support subblocks.


In [ ]:
def matrix_entry(mat, i, j):
    return float(mat[int(i), int(j)])

for patch_id, (patch, res_local, jac_local) in enumerate(
    zip(support_patches, local_residuals, local_jacobians)
):
    # for i, d in enumerate(patch.core_dofs[:20]):
    #     lv = float(res_local.vec[i])
    #     gv = float(res_global[d])
    #     print(f"i={i:3d}, global dof={d:3d}, local={lv:+.16e}, global={gv:+.16e}, diff={lv-gv:+.3e}")

    jac_err = 0.0
    for i, gi in enumerate(patch.core_dofs):
        for j, gj in enumerate(patch.core_dofs):
            jac_err = max(
                jac_err,
                abs(matrix_entry(jac_local.mat, i, j) - matrix_entry(jac_global, gi, gj)),
            )

    res_err = max(
        abs(float(res_local.vec[i]) - float(res_global[d]))
        for i, d in enumerate(patch.core_dofs)
    )

    print(f"Omega_{patch_id}:")
    print("  core elements     :", len(patch.core_elements))
    print("  support elements  :", len(patch.support_elements))
    print("  local/core dofs   :", len(patch.core_dofs))
    print("  Jacobian inf error:", f"{jac_err:.3e}")
    print("  residual inf error:", f"{res_err:.3e}")

    if not math.isclose(jac_err, 0.0, abs_tol=1e-10):
        raise RuntimeError("local nonlinear Jacobian does not match global core-core restriction")
    if not math.isclose(res_err, 0.0, abs_tol=1e-10):
        raise RuntimeError("local nonlinear residual does not match global core restriction")
